# Pembangunan Vector Database: Chunking, Embedding, dan ChromaDB

## Latar Belakang

Sistem RAG bekerja dengan prinsip mencari teks yang relevan terlebih dahulu, lalu menyerahkannya kepada LLM untuk menyusun jawaban. Agar pencarian dapat dilakukan berdasarkan makna (bukan sekadar kecocokan kata kunci), dibutuhkan sebuah vector database sebagai basis pengetahuan sistem. Tahap ini membangun basis pengetahuan tersebut dari artikel yang telah dibersihkan dan dilengkapi metadata kategori serta sentimen pada tahap sebelumnya.

## Vector Database dan Komponennya

Vector database adalah sistem penyimpanan yang dirancang untuk menyimpan data dalam bentuk representasi vektor numerik (embedding) dan melakukan pencarian berdasarkan kedekatan makna. Berbeda dengan pencarian berbasis kata kunci, vector database membandingkan vektor sehingga teks dengan makna serupa tetap dapat ditemukan meskipun menggunakan kata yang berbeda. Untuk data berjumlah besar, sistem ini menyediakan pencarian tetangga terdekat (nearest neighbor) yang efisien melalui algoritma seperti HNSW, kemampuan penyaringan berdasarkan metadata, serta persistensi data ke disk.

Setiap entri dalam vector database umumnya terdiri dari tiga komponen: vektor (representasi numerik yang digunakan untuk pencarian semantik), dokumen (teks asli dari potongan artikel), dan metadata (informasi tambahan seperti kategori, sentimen, judul, dan URL yang dapat digunakan untuk penyaringan maupun penelusuran sumber).

## ChromaDB

ChromaDB adalah salah satu vector database yang bersifat ringan, mudah digunakan, dan dapat berjalan secara lokal tanpa memerlukan server terpisah, sehingga cocok untuk penelitian dan prototipe. ChromaDB menyimpan ketiga komponen di atas (vektor, dokumen, dan metadata) untuk setiap entri, mendukung pencarian berbasis kedekatan dengan metrik yang dapat dikonfigurasi, serta menyediakan penyimpanan persisten sehingga data tidak hilang ketika sesi ditutup. Pada penelitian ini, ChromaDB digunakan sebagai vector database utama untuk menampung seluruh potongan artikel beserta metadatanya.

## Tujuan dan Alur

Tahap ini bertujuan membangun vector database dari artikel berlabel (`articles_with_sentiment.csv`) melalui empat langkah utama, yaitu memecah artikel menjadi potongan (chunk), mengubah tiap potongan menjadi vektor (embedding), menyimpan vektor beserta metadatanya ke ChromaDB, lalu menguji pencariannya. Hasil akhirnya adalah folder `chroma_db/` berisi seluruh artikel yang siap dicari secara semantik, lengkap dengan metadata kategori dan sentimen untuk keperluan penyaringan. Proses ini tidak mewajibkan GPU karena pembuatan embedding untuk 1.840 artikel berlangsung cepat, dan database disimpan secara permanen ke disk.

In [1]:
!pip install -q sentence-transformers chromadb pandas

## Persiapan dan Konfigurasi

In [2]:
import re
from pathlib import Path
import pandas as pd

def find_project_root():
    cwd = Path.cwd().resolve()
    if (cwd / "data").exists() and (cwd / "notebooks").exists():
        return cwd
    for parent in cwd.parents:
        if (parent / "data").exists() and (parent / "notebooks").exists():
            return parent
    return cwd

PROJECT_ROOT = find_project_root()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CHROMA_DIR = PROJECT_ROOT / "chroma_db"
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

# --- Konfigurasi ---
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
COLLECTION_NAME = "news_articles"
CHUNK_SIZE_WORDS = 200      # target maksimum kata per chunk
OVERLAP_SENTENCES = 1       # jumlah kalimat overlap antar chunk

print(f"Project root : {PROJECT_ROOT}")
print(f"ChromaDB dir : {CHROMA_DIR}")
print(f"Embedding    : {EMBEDDING_MODEL}")

Project root : C:\Users\user\news-rag-project
ChromaDB dir : C:\Users\user\news-rag-project\chroma_db
Embedding    : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Tahap ini menyiapkan jalur berkas serta parameter utama yang dijadikan variabel di awal agar mudah diubah dan konsisten digunakan. Lokasi root proyek dideteksi secara otomatis, dengan `data/processed` sebagai sumber data berlabel dan folder `chroma_db/` sebagai lokasi penyimpanan vector database. Konfigurasi utama mencakup model embedding yang digunakan (`paraphrase-multilingual-MiniLM-L12-v2`, model multilingual yang sesuai untuk bahasa Indonesia), nama koleksi (`news_articles`), serta parameter chunking berupa target maksimum 200 kata per potongan dengan overlap satu kalimat antarpotongan.

## Pemuatan Artikel

In [3]:
df = pd.read_csv(DATA_PROCESSED / "articles_with_sentiment.csv")
print(f"Total artikel: {len(df)}")
print(f"Kolom: {list(df.columns)}")
print(f"\nKategori: {df['kategori'].value_counts().to_dict()}")
print(f"Sentiment: {df['sentiment'].value_counts().to_dict()}")

Total artikel: 1840
Kolom: ['article_id', 'judul', 'isi', 'kategori', 'url', 'tanggal', 'n_words', 'sentiment', 'sentiment_score']

Kategori: {'news': 397, 'finance': 375, 'sport': 366, 'inet': 359, 'health': 343}
Sentiment: {'netral': 898, 'negatif': 494, 'positif': 448}


Data berlabel dimuat dari `articles_with_sentiment.csv`, berisi 1.840 artikel lengkap dengan metadata kategori dan sentimen yang akan disematkan ke setiap potongan. Distribusi kategori tetap relatif seimbang (343 hingga 397 artikel per kategori), sedangkan distribusi sentimen didominasi netral (898), diikuti negatif (494) dan positif (448). Kedua metadata inilah yang nantinya memungkinkan penyaringan pada saat pencarian di sistem RAG.

## Konsep Chunking: Mengapa Artikel Dipecah?

Artikel utuh tidak langsung dimasukkan ke vector database, melainkan dipecah terlebih dahulu menjadi potongan-potongan kecil (chunk). Pemecahan ini dilakukan atas tiga pertimbangan. Pertama, presisi retrieval: jika satu artikel direpresentasikan oleh satu vektor, vektor tersebut menjadi "rata-rata" makna seluruh artikel sehingga kabur, sedangkan dengan chunk kecil pencarian dapat menemukan bagian yang benar-benar relevan. Kedua, batas panjang masukan: baik model embedding maupun LLM memiliki batas jumlah token, sehingga potongan kecil lebih aman diproses. Ketiga, efisiensi konteks LLM: hanya chunk relevan yang dikirim ke LLM pada tahap generation, sehingga lebih hemat token dan menghasilkan jawaban yang lebih fokus.

Pemecahan dilakukan dengan beberapa pertimbangan praktik terbaik. Ukuran chunk dijaga tidak terlalu kecil (agar konteks tidak hilang) maupun terlalu besar (agar tidak kabur); untuk teks berita, kisaran 150-250 kata dinilai sesuai, dan penelitian ini menggunakan 200 kata. Antarchunk yang bersebelahan diberi tumpang-tindih (overlap) satu kalimat agar konteks pada batas potongan tidak terputus. Pemecahan juga bersifat sentence-aware, yaitu dilakukan pada akhir kalimat dan tidak memotong di tengah kalimat, sehingga makna tiap potongan tetap utuh. Dengan pendekatan ini, artikel pendek (di bawah 200 kata) tetap menjadi satu chunk, sedangkan artikel panjang dipecah menjadi beberapa chunk.

In [4]:
def split_sentences(text):
    """Pecah teks jadi kalimat (sederhana: pisah di . ! ?)."""
    parts = re.split(r"(?<=[.!?])\s+", str(text).strip())
    return [p.strip() for p in parts if p.strip()]

def chunk_text(text, chunk_size_words=CHUNK_SIZE_WORDS, overlap_sentences=OVERLAP_SENTENCES):
    """Gabung kalimat jadi chunk <= chunk_size_words, dengan overlap antar chunk."""
    sentences = split_sentences(text)
    chunks, cur, cur_words = [], [], 0
    for sent in sentences:
        w = len(sent.split())
        if cur and cur_words + w > chunk_size_words:
            chunks.append(" ".join(cur))
            cur = cur[-overlap_sentences:] if overlap_sentences > 0 else []  # bawa overlap
            cur_words = sum(len(s.split()) for s in cur)
        cur.append(sent)
        cur_words += w
    if cur:
        chunks.append(" ".join(cur))
    return chunks

# Contoh di 1 artikel
contoh = df.iloc[0]
contoh_chunks = chunk_text(contoh["isi"])
print(f"Artikel '{contoh['judul'][:50]}' ({len(contoh['isi'].split())} kata)")
print(f"-> dipecah jadi {len(contoh_chunks)} chunk")
for i, ch in enumerate(contoh_chunks):
    print(f"\n  [chunk {i}] ({len(ch.split())} kata): {ch[:120]}...")

Artikel 'Prancis Dilanda Gelombang Panas, 7 Orang Tewas' (266 kata)
-> dipecah jadi 2 chunk

  [chunk 0] (188 kata): Gelombang panas melanda wilayah Eropa bagian barat, termasuk Prancis . Pemerintah Prancis melaporkan sedikitnya tujuh or...

  [chunk 1] (97 kata): Suhu tinggi pada Senin (25/5) waktu setempat mendorong banyak orang mendatangi pantai-pantai di Prancis untuk mendingink...


Sebagai ilustrasi, sebuah artikel sepanjang 266 kata dipecah menjadi 2 chunk. Potongan pertama memuat sekitar 188 kata dan potongan kedua 97 kata, dengan pemecahan dilakukan pada batas kalimat dan satu kalimat di antaranya tumpang-tindih sebagai penjaga konteks. Hasil ini menunjukkan fungsi chunking bekerja sesuai rancangan, yaitu memecah artikel panjang secara rapi tanpa memotong kalimat.

### Terapkan chunking ke semua artikel

In [5]:
records = []
for _, row in df.iterrows():
    for idx, ch in enumerate(chunk_text(row["isi"])):
        records.append({
            "chunk_id": f"{row['article_id']}_{idx}",
            "text": ch,
            "article_id": str(row["article_id"]),
            "chunk_index": idx,
            "judul": str(row["judul"]),
            "kategori": str(row["kategori"]),
            "sentiment": str(row["sentiment"]),
            "sentiment_score": float(row["sentiment_score"]),
            "url": str(row["url"]),
            "tanggal": str(row["tanggal"]) if pd.notna(row["tanggal"]) else "unknown",
        })

chunks_df = pd.DataFrame(records)
print(f"Total chunk: {len(chunks_df)} (dari {len(df)} artikel)")
print(f"Rata-rata chunk per artikel: {len(chunks_df)/len(df):.2f}")
print(f"Rata-rata kata per chunk    : {chunks_df['text'].str.split().str.len().mean():.0f}")
print(f"\nDistribusi kategori (chunk): {chunks_df['kategori'].value_counts().to_dict()}")

Total chunk: 4448 (dari 1840 artikel)
Rata-rata chunk per artikel: 2.42
Rata-rata kata per chunk    : 158

Distribusi kategori (chunk): {'news': 993, 'inet': 953, 'finance': 930, 'health': 880, 'sport': 692}


Proses chunking diterapkan ke seluruh artikel, dan setiap chunk yang dihasilkan dibekali metadata dari artikel asalnya, yaitu `article_id`, `chunk_index`, judul, kategori, sentimen, skor sentimen, URL, dan tanggal. Metadata inilah yang nantinya memungkinkan penyaringan pada tahap pencarian (misalnya membatasi hasil pada kategori atau sentimen tertentu).

Dari 1.840 artikel dihasilkan 4.448 chunk, dengan rata-rata 2,42 chunk per artikel dan sekitar 158 kata per chunk. Angka ini sesuai harapan: artikel pendek tetap menjadi satu chunk, sementara artikel panjang terpecah menjadi beberapa bagian. Distribusi chunk antarkategori mencerminkan panjang artikelnya, di mana news dan inet menghasilkan chunk terbanyak (cenderung lebih panjang) sedangkan sport paling sedikit (cenderung lebih ringkas).

## Konsep Embedding: Teks Menjadi Vektor

Embedding adalah proses mengubah teks menjadi vektor angka (pada model ini berdimensi 384) yang menangkap maknanya. Teks dengan makna serupa akan memiliki vektor yang berdekatan dalam ruang 384 dimensi, meskipun kata-katanya berbeda; sebagai contoh, "saham naik" dan "IHSG menguat" menghasilkan vektor yang berdekatan. Kemampuan inilah yang memungkinkan pencarian bersifat semantik (berdasarkan makna), bukan sekadar kecocokan kata.

Beberapa praktik terbaik diterapkan dalam pembuatan embedding. Model yang sama harus digunakan untuk mengindeks chunk maupun untuk memproses query nantinya, agar vektor yang dihasilkan sebanding (model berbeda menghasilkan ruang vektor berbeda sehingga pencarian menjadi tidak valid). Model yang dipilih bersifat multilingual (`paraphrase-multilingual-MiniLM-L12-v2`) karena artikel berbahasa Indonesia, sehingga lebih sesuai dibandingkan model default berbahasa Inggris. Selain itu, vektor dinormalisasi agar dapat menggunakan cosine similarity, yaitu ukuran kemiripan arah vektor yang menjadi standar dalam retrieval teks.

In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Model loaded. Dimensi embedding: {model.get_sentence_embedding_dimension()}")

# Encode semua chunk (normalize biar cosine)
embeddings = model.encode(
    chunks_df["text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=64,
)
print(f"\nShape embeddings: {embeddings.shape}  (n_chunk x 384)")
print(f"Contoh 1 vektor (5 angka pertama): {embeddings[0][:5]}")

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

Model loaded. Dimensi embedding: 384


Batches:   0%|          | 0/70 [00:00<?, ?it/s]


Shape embeddings: (4448, 384)  (n_chunk x 384)
Contoh 1 vektor (5 angka pertama): [-0.01185396  0.0014525   0.03683236  0.02954438  0.07099485]


Model embedding dimuat dan menghasilkan vektor berdimensi 384. Seluruh 4.448 chunk kemudian dikodekan menjadi vektor ternormalisasi, menghasilkan matriks berukuran 4.448 x 384 (jumlah chunk x dimensi vektor). Vektor-vektor inilah yang akan disimpan ke dalam vector database untuk pencarian semantik.

## Pembuatan Koleksi ChromaDB

Setelah ribuan vektor terbentuk, dibutuhkan mekanisme untuk mencari vektor yang paling dekat dengan vektor query secara cepat meskipun datanya banyak, peran yang dijalankan vector database. Pendekatan sederhana menggunakan array biasa (numpy) memang memungkinkan untuk data kecil, tetapi vector database memberikan keunggulan berupa pencarian tetangga terdekat yang cepat melalui algoritma HNSW/ANN, penyaringan berdasarkan metadata, serta persistensi data ke disk sehingga tidak hilang ketika sesi ditutup.

Pembuatan koleksi pada ChromaDB mengikuti beberapa praktik terbaik. Koleksi dibuat melalui `PersistentClient` agar tersimpan permanen pada folder `chroma_db/`, dengan metrik kemiripan diset ke cosine (`hnsw:space = cosine`) yang sesuai dengan embedding ternormalisasi. Penyisipan data nantinya dilakukan secara batch agar efisien, dan metadata yang disimpan dibuat memadai untuk menampilkan sumber (judul dan URL) pada hasil pencarian.

In [7]:
import chromadb

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Kalau sudah ada (re-run), hapus dulu biar fresh
try:
    client.delete_collection(COLLECTION_NAME)
    print("Collection lama dihapus (re-run).")
except Exception:
    pass

# Buat collection dengan metrik cosine
collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)
print(f"Collection '{COLLECTION_NAME}' dibuat (cosine).")

Collection 'news_articles' dibuat (cosine).


Koleksi `news_articles` berhasil dibuat dengan metrik cosine. Jika koleksi sebelumnya sudah ada, koleksi lama dihapus terlebih dahulu agar pembangunan database selalu dimulai dari kondisi yang bersih saat dijalankan ulang.

### Masukin semua chunk ke ChromaDB

In [8]:
meta_cols = ["article_id", "chunk_index", "judul", "kategori",
             "sentiment", "sentiment_score", "url", "tanggal"]

ids = chunks_df["chunk_id"].tolist()
docs = chunks_df["text"].tolist()
metas = chunks_df[meta_cols].to_dict("records")
embs = embeddings.tolist()

B = 500
for i in range(0, len(ids), B):
    collection.add(
        ids=ids[i:i+B],
        embeddings=embs[i:i+B],
        documents=docs[i:i+B],
        metadatas=metas[i:i+B],
    )
print(f"Selesai. Total item di collection: {collection.count()}")

Selesai. Total item di collection: 4448


Seluruh chunk dimasukkan ke dalam koleksi melalui empat komponen yang dibutuhkan ChromaDB, yaitu `ids` (penanda unik tiap chunk), `embeddings` (vektor), `documents` (teks chunk asli), dan `metadatas` (metadata kategori, sentimen, judul, URL, dan lainnya). Penyisipan dilakukan secara batch sebesar 500 item per proses agar lebih efisien dibandingkan menyisipkan satu per satu. Seluruh 4.448 chunk berhasil dimasukkan ke koleksi, sehingga vector database kini lengkap dan siap digunakan untuk pencarian.

## Konsep Retrieval dan Penyaringan Metadata

Pencarian dilakukan dengan mengubah query menjadi vektor menggunakan model embedding yang sama, kemudian ChromaDB mencari chunk dengan vektor terdekat berdasarkan cosine. Hasil pencarian disertai nilai `distances`, di mana semakin kecil nilainya semakin mirip (cosine distance = 1 - cosine similarity).

Selain pencarian semantik, ChromaDB mendukung penyaringan berdasarkan metadata melalui argumen `where`, baik untuk satu syarat (misalnya membatasi pada kategori tertentu) maupun kombinasi beberapa syarat (misalnya kategori dan sentimen sekaligus). Kemampuan inilah yang menjadi inti integrasi sistem: komponen klasifikasi (kategori) dan analisis sentimen terhubung ke RAG, sehingga pencarian tidak sekadar menemukan chunk yang mirip, melainkan chunk yang mirip dalam kategori dan sentimen tertentu.

### Pencarian Semantik (Tanpa Filter)

In [9]:
def search(query, k=3, where=None):
    """Embed query (model sama) lalu cari di ChromaDB. Optional filter metadata."""
    q_emb = model.encode([query], normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=k, where=where)
    return res

def show(res, title):
    print("=" * 75)
    print(title)
    print("=" * 75)
    for i in range(len(res["ids"][0])):
        m = res["metadatas"][0][i]
        dist = res["distances"][0][i]
        doc = res["documents"][0][i]
        print(f"\n  #{i+1} | dist={dist:.3f} | [{m['kategori']}/{m['sentiment']}] {m['judul'][:55]}")
        print(f"      {doc[:130]}...")
    print()

# (a) Tanpa filter - murni semantik
show(search("kabar ekonomi dan pergerakan saham IHSG", k=3),
     "QUERY: 'kabar ekonomi dan pergerakan saham IHSG' (tanpa filter)")

QUERY: 'kabar ekonomi dan pergerakan saham IHSG' (tanpa filter)

  #1 | dist=0.326 | [finance/netral] TAPG Bagikan Dividen Rp 180/Saham, GOTO Masuk Sorotan M
      Pergerakan Indeks Harga Saham Gabungan (IHSG) masih berada di bawah tekanan pada perdagangan Selasa (26/5). IHSG ditutup turun 1,2...

  #2 | dist=0.344 | [finance/netral] IHSG Terbang di Sesi I, Saham Prajogo Pangestu Pesta Po
      Indeks Harga Saham Gabungan (IHSG) menguat pada perdagangan sesi I hari ini, Jumat (29/5). Indeks saham siang ini kuat ditopang se...

  #3 | dist=0.365 | [finance/negatif] Purbaya: 6 Bulan Lagi Orang Susah Makin Berkurang
      Menteri Keuangan Purbaya Yudhi Sadewa menyatakan percepatan pertumbuhan ekonomi akan digenjot pemerintah. Dia meramal enam bulan k...



Pada pengujian pertama, query "kabar ekonomi dan pergerakan saham IHSG" dijalankan tanpa filter untuk menguji pencarian semantik murni. Ketiga hasil teratas seluruhnya merupakan artikel kategori finance yang membahas IHSG, saham, dan ekonomi, dengan jarak yang kecil (0,326 hingga 0,365). Hasil ini menunjukkan pencarian berbasis makna bekerja dengan tepat, menemukan artikel yang relevan secara topik meskipun tidak semua kata pada query muncul persis di teks.

### Pencarian dengan Penyaringan Metadata

In [10]:
# (b) Filter kategori: cuma artikel teknologi (inet)
show(search("gadget dan teknologi terbaru", k=3, where={"kategori": "inet"}),
     "QUERY: 'gadget dan teknologi terbaru' (filter kategori=inet)")

# (c) Filter kombinasi: sport YANG positif
show(search("pertandingan olahraga Indonesia", k=3,
            where={"$and": [{"kategori": "sport"}, {"sentiment": "positif"}]}),
     "QUERY: 'pertandingan olahraga Indonesia' (filter sport + positif)")

QUERY: 'gadget dan teknologi terbaru' (filter kategori=inet)

  #1 | dist=0.418 | [inet/positif] Erafone Hadir Lebih Dekat Lewat Pengalaman Belanja Gadg
      Perangkat digital kini telah menjadi bagian penting dalam aktivitas sehari-hari masyarakat. Mulai dari bekerja, belajar, hiburan, ...

  #2 | dist=0.484 | [inet/netral] Gadget Masa Depan Apple: Ada AirPods Berkamera AI
      Sederet bocoran terbaru baru saja mengungkap rencana ambisius Apple untuk lini produk masa depannya. Raksasa asal Cupertino ini di...

  #3 | dist=0.492 | [inet/positif] Erafone Hadir Lebih Dekat Lewat Pengalaman Belanja Gadg
      Melalui kampanye "Erafone Lebih Dekat", erafone ingin menghadirkan pengalaman belanja gadget yang lebih praktis dan mudah dijangka...

QUERY: 'pertandingan olahraga Indonesia' (filter sport + positif)

  #1 | dist=0.265 | [sport/positif] Menjaring Atlet Asian Games 2026 di Kejurnas Akuatik
      Pengurus Besar Akuatik Indonesia (PB AI) menggelar Kejuaraan Nasional Akuatik Indonesia

Dua pengujian berikutnya menerapkan penyaringan metadata. Pada penyaringan satu syarat (kategori = inet), query "gadget dan teknologi terbaru" menghasilkan tiga chunk yang seluruhnya berasal dari kategori inet. Pada penyaringan kombinasi (kategori = sport dan sentimen = positif), query "pertandingan olahraga Indonesia" menghasilkan tiga chunk yang seluruhnya merupakan berita olahraga bersentimen positif dengan jarak yang sangat kecil (0,265 hingga 0,281). Kedua hasil ini membuktikan penyaringan metadata berfungsi dengan tepat, baik untuk syarat tunggal maupun gabungan.

Pada hasil penyaringan kategori inet, terlihat satu artikel ("Erafone Hadir Lebih Dekat") muncul dua kali sebagai chunk yang berbeda. Hal ini wajar karena artikel panjang terpecah menjadi beberapa chunk, namun berpotensi membuat sumber yang sama terhitung ganda. Temuan ini menjadi dasar penerapan deduplikasi per artikel pada tahap pipeline RAG agar konteks yang dikirim ke LLM lebih efisien dan sumber tidak berulang.

### Verifikasi Penyaringan

In [11]:
res = search("berita kesehatan", k=5, where={"kategori": "health"})
kategori_hasil = [m["kategori"] for m in res["metadatas"][0]]
print(f"Query difilter kategori=health -> kategori hasil: {set(kategori_hasil)}")
assert all(k == "health" for k in kategori_hasil), "Filter gagal!"
print("OK: filter metadata bekerja. Semua hasil dari kategori yang diminta.")

Query difilter kategori=health -> kategori hasil: {'health'}
OK: filter metadata bekerja. Semua hasil dari kategori yang diminta.


Sebagai pemeriksaan akhir, dilakukan verifikasi otomatis untuk memastikan penyaringan benar-benar membatasi hasil sesuai metadata yang diminta. Query "berita kesehatan" dengan filter kategori = health diperiksa, dan seluruh hasil yang dikembalikan terbukti berasal dari kategori health. Verifikasi ini menegaskan bahwa mekanisme penyaringan metadata pada ChromaDB bekerja secara andal, yang menjadi fondasi penting bagi integrasi komponen klasifikasi dan sentimen ke dalam sistem RAG.

Kita telah membangun vector database sebagai basis pengetahuan sistem RAG. Sebanyak 1.840 artikel dipecah menjadi 4.448 chunk, dikodekan menjadi vektor berdimensi 384, lalu disimpan ke ChromaDB beserta metadata kategori dan sentimennya, tersimpan permanen pada folder `chroma_db/`. Pengujian menunjukkan pencarian semantik menghasilkan artikel yang relevan, dan penyaringan metadata bekerja dengan tepat baik untuk syarat tunggal maupun gabungan. Dengan basis pengetahuan yang siap dicari ini, tahap berikutnya merangkai seluruh komponen menjadi pipeline RAG yang utuh, yaitu mengklasifikasikan pertanyaan, mengambil chunk relevan, lalu menghasilkan jawaban menggunakan LLM.